# LIC — Evidencia BI vs SQL (Validación técnica v9)

✅ **Patch de cierre para el caso UTM vs “Moneda revisar”**

Tu evidencia DW_SEM confirma que **no existe `moneda_norm='UTM'`** en `dw_sem.v_fact_licitaciones_sem_v2`.  
Los 590 registros corresponden a:

- `moneda = 'Moneda revisar'`
- `moneda_norm = NULL`

Por tanto, la auditoría debe:
- Auditar por **`moneda_norm`** para CLP/USD/CLF/EUR
- Auditar por **`moneda='Moneda revisar'`** para los 590 registros
- **Eliminar UTM** como categoría de auditoría (o mapearlo explícitamente a “Moneda revisar” si tu dashboard lo rotula así)

Outputs:
- `lic_kpis_sql.csv`
- `lic_kpis_sql.log`
- `lic_kpis_bi_vs_sql.csv`


## 1) Setup

In [1]:
import os, math
import datetime as dt
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
from sqlalchemy.exc import OperationalError

## 2) Conexión PostgreSQL

In [2]:
PG_HOST = os.getenv("PG_HOST", "localhost")
PG_PORT = int(os.getenv("PG_PORT", "5433"))
PG_DB   = os.getenv("PG_DB", "chilecompra")
PG_USER = os.getenv("PG_USER", "chile_user")
PG_PASS = os.getenv("PG_PASS", "CHANGE_ME")

def make_engine():
    if PG_PASS == "":
        raise ValueError("PG_PASS vacío. Define PG_PASS como variable de entorno.")
    url = f"postgresql+psycopg2://{PG_USER}:{PG_PASS}@{PG_HOST}:{PG_PORT}/{PG_DB}"
    return create_engine(url, pool_pre_ping=True)

engine = None

def test_connection():
    global engine
    engine = make_engine()
    try:
        with engine.connect() as conn:
            v = conn.execute(text("SELECT 1 AS ok;")).fetchone()[0]
        print("✅ Conexión OK:", v)
        return True
    except OperationalError as e:
        print("❌ OperationalError. Revisa PG_HOST/PG_PORT/PG_PASS.")
        print(str(e)[:600], "...")
        return False

test_connection()

✅ Conexión OK: 1


True

## 3) Parámetros de auditoría (moneda_norm + excepción Moneda revisar)

In [3]:
from pathlib import Path

# Monedas auditables por norma (según DW_SEM)
MONEDAS_NORM = ["CLP","USD","EUR","CLF"]  # UTM se excluye porque no existe en moneda_norm

# Categoría especial (según DAX / SEM)
MONEDA_ESPECIAL = "Moneda revisar"

# Flags DQ usados por DAX en Monto Estimado
APLICAR_DQ_ESTIMADO = True

BASE_DIR = Path.cwd()
OUT_DIR = Path(os.getenv("OUT_DIR", BASE_DIR / "outputs"))
OUT_DIR.mkdir(parents=True, exist_ok=True)


def fetch_df(sql: str) -> pd.DataFrame:
    if engine is None:
        raise RuntimeError("Engine no inicializado. Ejecuta test_connection() primero.")
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn)

def dq_estimado_where():
    if not APLICAR_DQ_ESTIMADO:
        return ""
    return " AND flag_monto_estimado_anom = 0 AND flag_moneda_missing = 0 AND flag_moneda_unknown = 0 "

print("OUT_DIR =", OUT_DIR)

OUT_DIR = /home/engineer/Documents/Proyectos/TFM/docs/evidence/fase7_auditoria_BI/Evidencia_BI_vs_SQL/LIC/outputs


## 4) Diagnóstico (moneda vs moneda_norm)

In [4]:
sql_diag = '''
SELECT 
  moneda,
  moneda_norm,
  COUNT(*) AS n_rows,
  COUNT(DISTINCT licitacion_bk) AS n_lic_bk,
  COALESCE(SUM(monto_adjudicado),0) AS sum_adj
FROM dw_sem.v_fact_licitaciones_sem_v2
GROUP BY 1,2
ORDER BY n_rows DESC;
'''
df_diag = fetch_df(sql_diag)
df_diag

,moneda,moneda_norm,n_rows,n_lic_bk,sum_adj
0,Peso Chileno,CLP,4620401,4620401,4.690131e+14
1,Dolar,USD,7329,7329,1.633984e+10
2,Unidad de Fomento,CLF,4761,4761,1.119013e+11
3,Moneda revisar,None,590,590,1.645427e+09


## 5) Plan de KPIs

In [5]:
def build_plan():
    rows = []
    # Globales (sin filtro)
    rows += [
        {"kpi":"LIC_Conteo_Global", "moneda_val": None},
        {"kpi":"LIC_Proveedores_Distintos_Global", "moneda_val": None},
        {"kpi":"LIC_Productos_ONU_Distintos_Global", "moneda_val": None},
    ]
    # Por moneda_norm
    for m in MONEDAS_NORM:
        rows += [
            {"kpi":"LIC_Conteo", "moneda_val": m},
            {"kpi":"LIC_Monto_Estimado", "moneda_val": m},
            {"kpi":"LIC_Monto_Adjudicado", "moneda_val": m},
            {"kpi":"LIC_Proveedores_Distintos", "moneda_val": m},
            {"kpi":"LIC_Productos_ONU_Distintos", "moneda_val": m},
        ]
    # Moneda especial
    rows += [
        {"kpi":"LIC_Conteo", "moneda_val": MONEDA_ESPECIAL},
        {"kpi":"LIC_Monto_Estimado", "moneda_val": MONEDA_ESPECIAL},
        {"kpi":"LIC_Monto_Adjudicado", "moneda_val": MONEDA_ESPECIAL},
        {"kpi":"LIC_Proveedores_Distintos", "moneda_val": MONEDA_ESPECIAL},
        {"kpi":"LIC_Productos_ONU_Distintos", "moneda_val": MONEDA_ESPECIAL},
    ]
    return pd.DataFrame(rows)

df_plan = build_plan()
df_plan

,kpi,moneda_val
0,LIC_Conteo_Global,None
1,LIC_Proveedores_Distintos_Global,None
2,LIC_Productos_ONU_Distintos_Global,None
3,LIC_Conteo,CLP
4,LIC_Monto_Estimado,CLP
5,LIC_Monto_Adjudicado,CLP
6,LIC_Proveedores_Distintos,CLP
7,LIC_Productos_ONU_Distintos,CLP
8,LIC_Conteo,USD
9,LIC_Monto_Estimado,USD


## 6) SQL builders (réplica DAX + excepción Moneda revisar)

In [6]:
def where_moneda(moneda_val):
    if moneda_val is None:
        return ""
    if moneda_val == MONEDA_ESPECIAL:
        return f" WHERE moneda = '{MONEDA_ESPECIAL}' "
    # caso normal
    return f" WHERE moneda_norm = '{moneda_val}' "

def sql_for_kpi(kpi: str, moneda_val):
    if kpi == "LIC_Conteo_Global":
        return "SELECT COUNT(DISTINCT licitacion_bk)::bigint AS valor FROM dw_sem.v_fact_licitaciones_sem_v2;"
    if kpi == "LIC_Conteo":
        return "SELECT COUNT(DISTINCT licitacion_bk)::bigint AS valor FROM dw_sem.v_fact_licitaciones_sem_v2" + where_moneda(moneda_val) + ";"

    if kpi == "LIC_Proveedores_Distintos_Global":
        return "SELECT COUNT(DISTINCT proveedor_sk)::bigint AS valor FROM dw_sem.v_fact_licitaciones_sem_v2;"
    if kpi == "LIC_Proveedores_Distintos":
        return "SELECT COUNT(DISTINCT proveedor_sk)::bigint AS valor FROM dw_sem.v_fact_licitaciones_sem_v2" + where_moneda(moneda_val) + ";"

    if kpi == "LIC_Productos_ONU_Distintos_Global":
        return "SELECT COUNT(DISTINCT producto_onu_sk)::bigint AS valor FROM dw_sem.v_fact_licitaciones_sem_v2;"
    if kpi == "LIC_Productos_ONU_Distintos":
        return "SELECT COUNT(DISTINCT producto_onu_sk)::bigint AS valor FROM dw_sem.v_fact_licitaciones_sem_v2" + where_moneda(moneda_val) + ";"

    if kpi == "LIC_Monto_Adjudicado":
        return "SELECT COALESCE(SUM(monto_adjudicado),0)::numeric(38,0) AS valor FROM dw_sem.v_fact_licitaciones_sem_v2" + where_moneda(moneda_val) + ";"

    if kpi == "LIC_Monto_Estimado":
        # réplica DAX: SUMX(VALUES(licitacion_sk), MAX(monto_estimado) con flags)
        inner_where = where_moneda(moneda_val)
        if inner_where == "":
            inner_where = " WHERE 1=1 "
        else:
            inner_where = inner_where.replace(" WHERE ", " WHERE 1=1 AND ")
        inner_where += dq_estimado_where()

        inner = (
            "SELECT MAX(monto_estimado) AS mx "
            "FROM dw_sem.v_fact_licitaciones_sem_v2 "
            + inner_where +
            " GROUP BY licitacion_sk"
        )
        return f"SELECT COALESCE(SUM(mx),0)::numeric(38,0) AS valor FROM ({inner}) t;"

    raise ValueError("KPI no soportado: " + kpi)

print(sql_for_kpi("LIC_Conteo","CLP"))
print(sql_for_kpi("LIC_Conteo", MONEDA_ESPECIAL))

SELECT COUNT(DISTINCT licitacion_bk)::bigint AS valor FROM dw_sem.v_fact_licitaciones_sem_v2 WHERE moneda_norm = 'CLP' ;
SELECT COUNT(DISTINCT licitacion_bk)::bigint AS valor FROM dw_sem.v_fact_licitaciones_sem_v2 WHERE moneda = 'Moneda revisar' ;


## 7) Ejecutar SQL y guardar evidencia

In [7]:
run_ts = dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
rows = []
log_lines = [f"LIC KPIs SQL EVIDENCE - {run_ts}", "-"*60]

for _, r in df_plan.iterrows():
    kpi = r["kpi"]
    moneda_val = r["moneda_val"]
    sql = sql_for_kpi(kpi, moneda_val)

    log_lines.append(f"\n## KPI: {kpi}" + (f" | moneda: {moneda_val}" if moneda_val is not None else ""))
    log_lines.append(sql)

    dfq = fetch_df(sql)
    valor = None
    if len(dfq) and "valor" in dfq.columns:
        v = dfq.loc[0, "valor"]
        if v is not None and not (isinstance(v, float) and math.isnan(v)):
            try:
                valor = float(v)
            except Exception:
                valor = float(str(v))

    rows.append({"kpi": kpi, "moneda_val": moneda_val, "valor_sql": valor})
    log_lines.append(dfq.to_string(index=False))

df_sql = pd.DataFrame(rows)

sql_csv_path = os.path.join(OUT_DIR, "lic_kpis_sql.csv")
sql_log_path = os.path.join(OUT_DIR, "lic_kpis_sql.log")
df_sql.to_csv(sql_csv_path, index=False, encoding="utf-8")
with open(sql_log_path, "w", encoding="utf-8") as f:
    f.write("\n".join(log_lines))

print("✅ Guardado:", sql_csv_path)
print("✅ Guardado:", sql_log_path)

df_sql

✅ Guardado: /home/engineer/Documents/Proyectos/TFM/docs/evidence/fase7_auditoria_BI/Evidencia_BI_vs_SQL/LIC/outputs/lic_kpis_sql.csv
✅ Guardado: /home/engineer/Documents/Proyectos/TFM/docs/evidence/fase7_auditoria_BI/Evidencia_BI_vs_SQL/LIC/outputs/lic_kpis_sql.log


,kpi,moneda_val,valor_sql
0,LIC_Conteo_Global,None,4.633081e+06
1,LIC_Proveedores_Distintos_Global,None,1.951800e+04
2,LIC_Productos_ONU_Distintos_Global,None,8.319000e+03
3,LIC_Conteo,CLP,4.620401e+06
4,LIC_Monto_Estimado,CLP,3.723090e+14
5,LIC_Monto_Adjudicado,CLP,4.690131e+14
6,LIC_Proveedores_Distintos,CLP,1.942500e+04
7,LIC_Productos_ONU_Distintos,CLP,8.304000e+03
8,LIC_Conteo,USD,7.329000e+03
9,LIC_Monto_Estimado,USD,5.876034e+11


## 8) Comparación BI vs SQL

In [8]:
# ============================================================
# 8) Comparación BI vs SQL — BLOQUE COMPLETO (robusto)
# Requiere:
#   - df_sql con columnas: kpi, moneda_val, valor_sql
#   - OUT_DIR definido (Path o str)
# Produce:
#   - OUT_DIR/lic_kpis_bi_vs_sql.csv
# ============================================================

from pathlib import Path
import os, glob
import pandas as pd
import numpy as np

# -----------------------------
# 8.1 - Resolver OUT_DIR
# -----------------------------
out_dir = Path(OUT_DIR) if "OUT_DIR" in globals() else Path(os.getenv("OUT_DIR", "./outputs"))
out_dir.mkdir(parents=True, exist_ok=True)

print("OUT_DIR =", out_dir)
print("Existe OUT_DIR?", out_dir.exists())

# -----------------------------
# 8.2 - Cargar CSV BI (robusto)
# -----------------------------
bi_csv_path = out_dir / "lic_kpis_bi.csv"

# Lista CSVs disponibles (root y recursivo)
csvs_top = sorted(out_dir.glob("*.csv"))
csvs_all = sorted(out_dir.rglob("*.csv"))

print("\nCSVs en OUT_DIR (nivel raíz):", len(csvs_top))
for f in csvs_top[:30]:
    print(" -", f.name)

if len(csvs_all) != len(csvs_top):
    print("\nCSVs en OUT_DIR (recursivo):", len(csvs_all))
    for f in csvs_all[:30]:
        print(" -", f.relative_to(out_dir))

# Ranking de candidatos por nombre
def _score(p: Path) -> int:
    n = p.name.lower()
    s = 100
    if "lic_kpis_bi" in n: s -= 50
    if "kpi" in n:         s -= 20
    if "lic" in n:         s -= 10
    if "bi" in n:          s -= 5
    return s

candidatos = sorted(csvs_all, key=_score)

# Función de lectura CSV tolerante (sep , o ;)
def _read_csv_tolerant(path: Path) -> pd.DataFrame:
    # Intenta coma, luego punto y coma si detecta 1 columna con ';'
    df = pd.read_csv(path)
    if len(df.columns) == 1 and ";" in df.columns[0]:
        df = pd.read_csv(path, sep=";")
    return df

# Carga
if bi_csv_path.exists():
    df_bi = _read_csv_tolerant(bi_csv_path)
    src_bi = bi_csv_path
    print("\n✅ Cargado BI (nombre estándar):", bi_csv_path, "| filas:", len(df_bi))

elif len(candidatos) == 1:
    df_bi = _read_csv_tolerant(candidatos[0])
    src_bi = candidatos[0]
    print("\n⚠️ No existía lic_kpis_bi.csv. Se cargó único CSV encontrado:", candidatos[0], "| filas:", len(df_bi))

elif len(candidatos) > 1:
    print("\n⚠️ No existe lic_kpis_bi.csv. Se encontraron múltiples CSV candidatos (top 10):")
    for f in candidatos[:10]:
        print(" -", f.relative_to(out_dir))
    raise FileNotFoundError(
        "Ambigüedad: hay varios CSV posibles.\n"
        "Solución: copia/renombra el correcto como OUT_DIR/lic_kpis_bi.csv o deja SOLO uno en OUT_DIR."
    )
else:
    raise FileNotFoundError(
        f"No existe {bi_csv_path} y no se detectaron CSV en OUT_DIR.\n"
        f"Solución: guarda lic_kpis_bi.csv dentro de: {out_dir}"
    )

# -----------------------------
# 8.3 - Normalizar columnas BI
# -----------------------------
# Normaliza (lower + strip)
df_bi.columns = [c.strip().lower() for c in df_bi.columns]

# Mapeo flexible (export típico Power BI / variantes)
rename_map = {
    "kpi_name": "kpi",
    "indicador": "kpi",
    "nombre_kpi": "kpi",
    "moneda": "moneda_val",
    "currency": "moneda_val",
    "divisa": "moneda_val",
    "valor": "valor_pbi",
    "value": "valor_pbi",
    "monto": "valor_pbi",
    "importe": "valor_pbi",
    "valor_bi": "valor_pbi",
}
df_bi = df_bi.rename(columns=rename_map)

print("\nColumnas BI detectadas:", df_bi.columns.tolist(), "| fuente:", src_bi.name)


df_bi.columns = [c.strip().lower() for c in df_bi.columns]  # asegura normalización

# Si aún no existe 'valor_pbi', intenta inferirla
if "valor_pbi" not in df_bi.columns:
    # candidatos típicos: cualquier columna que contenga 'valor', 'monto', 'importe', 'pbi', 'bi'
    cand = [c for c in df_bi.columns if any(k in c for k in ["valor", "monto", "importe", "pbi", "bi", "value"])]

    # excluye columnas clave
    cand = [c for c in cand if c not in ["kpi", "moneda_val"]]

    if len(cand) == 1:
        df_bi = df_bi.rename(columns={cand[0]: "valor_pbi"})
        print(f"⚠️ Columna valor inferida: '{cand[0]}' -> 'valor_pbi'")
    else:
        # Diagnóstico útil
        print("❌ No se pudo inferir 'valor_pbi'. Columnas actuales:", df_bi.columns.tolist())
        print("Candidatos encontrados:", cand)
        raise ValueError(
            "El CSV BI no tiene columna 'valor_pbi' y no se pudo inferir automáticamente.\n"
            "Solución: renombra manualmente la columna de valor en tu CSV a 'valor_pbi'."
        )


# Validación requerida
cols_req = {"kpi", "moneda_val", "valor_pbi"}
faltan = cols_req - set(df_bi.columns)
if faltan:
    raise ValueError(
        f"El CSV BI NO cumple el contrato. Faltan: {faltan}. "
        f"Columnas actuales: {list(df_bi.columns)}. "
        f"Fuente BI usada: {src_bi}"
    )

# Normalización de valores clave
df_bi["kpi"] = df_bi["kpi"].astype(str).str.strip()
df_bi["moneda_val"] = df_bi["moneda_val"].astype(str).str.strip().str.upper()
df_bi["valor_pbi"] = pd.to_numeric(df_bi["valor_pbi"], errors="coerce")

# -----------------------------
# 8.4 - Validar df_sql (contrato)
# -----------------------------
req_sql = {"kpi", "moneda_val", "valor_sql"}
faltan_sql = req_sql - set(df_sql.columns)
if faltan_sql:
    raise ValueError(f"df_sql NO tiene columnas requeridas {req_sql}. Faltan: {faltan_sql}. Columnas: {list(df_sql.columns)}")

df_sql["kpi"] = df_sql["kpi"].astype(str).str.strip()
df_sql["moneda_val"] = df_sql["moneda_val"].astype(str).str.strip().str.upper()
df_sql["valor_sql"] = pd.to_numeric(df_sql["valor_sql"], errors="coerce")

# -----------------------------
# 8.5 - Merge + métricas de comparación
# -----------------------------
df_cmp = df_sql.merge(df_bi[["kpi", "moneda_val", "valor_pbi"]], on=["kpi", "moneda_val"], how="left")

df_cmp["diff_abs"] = df_cmp["valor_pbi"] - df_cmp["valor_sql"]

mask_pct = (df_cmp["valor_sql"].notna() & (df_cmp["valor_sql"] != 0) & df_cmp["valor_pbi"].notna())
df_cmp["diff_pct"] = np.where(mask_pct, (df_cmp["diff_abs"] / df_cmp["valor_sql"]) * 100, np.nan)

# Estados
df_cmp["estado"] = "OK"
df_cmp.loc[df_cmp["valor_pbi"].isna(), "estado"] = "FALTA_PBI"
df_cmp.loc[df_cmp["valor_sql"].isna(), "estado"] = "FALTA_SQL"

mask_no_aplica = (
    (df_cmp["moneda_val"].astype(str).str.upper() == "NONE") &
    (df_cmp["kpi"].astype(str).str.contains("_Global", na=False))
)

df_cmp.loc[mask_no_aplica, "estado"] = "NO_APLICA"
df_cmp.loc[mask_no_aplica, ["diff_abs", "diff_pct"]] = np.nan


# Caso SQL=0
df_cmp.loc[(df_cmp["estado"]=="OK") & (df_cmp["valor_sql"]==0) & (df_cmp["valor_pbi"]==0),
           ["diff_abs","diff_pct","estado"]] = [0, 0, "OK"]
df_cmp.loc[(df_cmp["estado"]=="OK") & (df_cmp["valor_sql"]==0) & (df_cmp["valor_pbi"]!=0),
           "estado"] = "NO_OK_SQL0"

# Diferencias != 0
df_cmp.loc[(df_cmp["estado"]=="OK") & df_cmp["diff_abs"].notna() & (df_cmp["diff_abs"]!=0),
           "estado"] = "NO_OK"

# -----------------------------
# 8.6 - Guardar salida + resumen
# -----------------------------
cmp_csv_path = out_dir / "lic_kpis_bi_vs_sql.csv"
df_cmp.to_csv(cmp_csv_path, index=False, encoding="utf-8")

print("\n✅ Guardado:", cmp_csv_path)
print("Resumen por estado:")
print(df_cmp["estado"].value_counts(dropna=False))

# Resultado ordenado (para display en notebook)
df_cmp = df_cmp.sort_values(["kpi", "moneda_val"], na_position="first")
df_cmp


OUT_DIR = /home/engineer/Documents/Proyectos/TFM/docs/evidence/fase7_auditoria_BI/Evidencia_BI_vs_SQL/LIC/outputs
Existe OUT_DIR? True

CSVs en OUT_DIR (nivel raíz): 3
 - lic_kpis_bi.csv
 - lic_kpis_bi_vs_sql.csv
 - lic_kpis_sql.csv

✅ Cargado BI (nombre estándar): /home/engineer/Documents/Proyectos/TFM/docs/evidence/fase7_auditoria_BI/Evidencia_BI_vs_SQL/LIC/outputs/lic_kpis_bi.csv | filas: 28

Columnas BI detectadas: ['kpi', 'moneda_val', 'valor_pbi'] | fuente: lic_kpis_bi.csv

✅ Guardado: /home/engineer/Documents/Proyectos/TFM/docs/evidence/fase7_auditoria_BI/Evidencia_BI_vs_SQL/LIC/outputs/lic_kpis_bi_vs_sql.csv
Resumen por estado:
OK           25
NO_APLICA     3
Name: estado, dtype: int64


,kpi,moneda_val,valor_sql,valor_pbi,diff_abs,diff_pct,estado
18,LIC_Conteo,CLF,4.761000e+03,4.761000e+03,0.0,0.0,OK
3,LIC_Conteo,CLP,4.620401e+06,4.620401e+06,0.0,0.0,OK
13,LIC_Conteo,EUR,0.000000e+00,0.000000e+00,0.0,0.0,OK
23,LIC_Conteo,MONEDA REVISAR,5.900000e+02,5.900000e+02,0.0,0.0,OK
8,LIC_Conteo,USD,7.329000e+03,7.329000e+03,0.0,0.0,OK
0,LIC_Conteo_Global,NONE,4.633081e+06,NaN,NaN,NaN,NO_APLICA
20,LIC_Monto_Adjudicado,CLF,1.119013e+11,1.119013e+11,0.0,0.0,OK
5,LIC_Monto_Adjudicado,CLP,4.690131e+14,4.690131e+14,0.0,0.0,OK
15,LIC_Monto_Adjudicado,EUR,0.000000e+00,0.000000e+00,0.0,0.0,OK
25,LIC_Monto_Adjudicado,MONEDA REVISAR,1.645427e+09,1.645427e+09,0.0,0.0,OK
